
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 03 - DataFrame Relational Operations in Spark

This demonstration shows how to effectively use joins and set operations with DataFrames, focusing on performance optimization and best practices.

### Objectives
- Understand different types of DataFrame joins
- Implement performance optimizations for joins
- Handle complex join scenarios
- Use set operations effectively
- Apply best practices for data skew

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Setup and Data Loading

First, let's load our sample retail data tables and examine their structures.

In [0]:
from pyspark.sql.functions import *

# Read the data 
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")
customers_df = spark.read.table("samples.bakehouse.sales_customers")
franchises_df = spark.read.table("samples.bakehouse.sales_franchises")
suppliers_df = spark.read.table("samples.bakehouse.sales_suppliers")

In [0]:
# Examine schemas
transactions_df.printSchema()

root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



In [0]:
customers_df.printSchema()

root
 |-- customerID: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email_address: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- postal_zip_code: long (nullable = true)
 |-- gender: string (nullable = true)



In [0]:
franchises_df.printSchema()

root
 |-- franchiseID: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- size: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- supplierID: long (nullable = true)



In [0]:
suppliers_df.printSchema()

root
 |-- supplierID: long (nullable = true)
 |-- name: string (nullable = true)
 |-- ingredient: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- size: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- approved: string (nullable = true)



## B. Basic Join Operations

Let's start with simple join operations to combine our data.

In [0]:
# Inner join example to enrich the transactions with store information
enriched_transactions = franchises_df.join(
    transactions_df,
    on="franchiseID",
    how="inner"
)

display(enriched_transactions)

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID,transactionID,customerID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2002961,1000253,2024-05-14T12:17:01.495952Z,Golden Gate Ginger,8,3,24,amex,378154478982993
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003007,1000226,2024-05-10T23:10:10.239954Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003017,1000108,2024-05-16T16:34:10.61372Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003068,1000173,2024-05-02T04:31:51.612094Z,Pearly Pies,28,3,84,amex,343808569426192
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003103,1000075,2024-05-04T23:44:26.902224Z,Pearly Pies,28,3,84,visa,4377080942201798
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003147,1000295,2024-05-15T16:17:06.25945Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003196,1000237,2024-05-07T11:13:22.469231Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003329,1000272,2024-05-06T03:32:16.017968Z,Outback Oatmeal,28,3,84,visa,4872480716880043
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2001264,1000209,2024-05-16T17:32:28.547589Z,Pearly Pies,28,3,84,mastercard,5287105980593305
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2001287,1000120,2024-05-15T08:41:28.406738Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


In [0]:
# The "on" clause can contain an expression
enriched_transactions = franchises_df.join(
    transactions_df,
    on= transactions_df.franchiseID == franchises_df.franchiseID,
    how="inner"
)

display(enriched_transactions)

# This is particularly useful if the join key is named differently in both entities

franchiseID,name,city,district,zipcode,country,size,longitude,latitude,supplierID,transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2002961,1000253,3000047,2024-05-14T12:17:01.495952Z,Golden Gate Ginger,8,3,24,amex,378154478982993
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003007,1000226,3000047,2024-05-10T23:10:10.239954Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003017,1000108,3000047,2024-05-16T16:34:10.61372Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003068,1000173,3000047,2024-05-02T04:31:51.612094Z,Pearly Pies,28,3,84,amex,343808569426192
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003103,1000075,3000047,2024-05-04T23:44:26.902224Z,Pearly Pies,28,3,84,visa,4377080942201798
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003147,1000295,3000047,2024-05-15T16:17:06.25945Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003196,1000237,3000047,2024-05-07T11:13:22.469231Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2003329,1000272,3000047,2024-05-06T03:32:16.017968Z,Outback Oatmeal,28,3,84,visa,4872480716880043
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2001264,1000209,3000047,2024-05-16T17:32:28.547589Z,Pearly Pies,28,3,84,mastercard,5287105980593305
3000047,Sweet Sinsations,Stockholm,Sodermalm,116 45,Sweden,S,18.072,59.3144,4000047,2001287,1000120,3000047,2024-05-15T08:41:28.406738Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


In [0]:
# Please note how all fields from both dataframes are present in the result, a better practice is to project the columns you need from each entity
# We will also alias some of the columns to disambiguate column names
enriched_transactions = franchises_df \
    .select(
        "franchiseID", 
        col("name").alias("store_name"), 
        col("city").alias("store_city"), 
        col("country").alias("store_country")
        ) \
    .join(
        transactions_df,
        on="franchiseID",
        how="inner"
    )
    
display(enriched_transactions)

franchiseID,store_name,store_city,store_country,transactionID,customerID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
3000047,Sweet Sinsations,Stockholm,Sweden,2002961,1000253,2024-05-14T12:17:01.495952Z,Golden Gate Ginger,8,3,24,amex,378154478982993
3000047,Sweet Sinsations,Stockholm,Sweden,2003007,1000226,2024-05-10T23:10:10.239954Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
3000047,Sweet Sinsations,Stockholm,Sweden,2003017,1000108,2024-05-16T16:34:10.61372Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
3000047,Sweet Sinsations,Stockholm,Sweden,2003068,1000173,2024-05-02T04:31:51.612094Z,Pearly Pies,28,3,84,amex,343808569426192
3000047,Sweet Sinsations,Stockholm,Sweden,2003103,1000075,2024-05-04T23:44:26.902224Z,Pearly Pies,28,3,84,visa,4377080942201798
3000047,Sweet Sinsations,Stockholm,Sweden,2003147,1000295,2024-05-15T16:17:06.25945Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
3000047,Sweet Sinsations,Stockholm,Sweden,2003196,1000237,2024-05-07T11:13:22.469231Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
3000047,Sweet Sinsations,Stockholm,Sweden,2003329,1000272,2024-05-06T03:32:16.017968Z,Outback Oatmeal,28,3,84,visa,4872480716880043
3000047,Sweet Sinsations,Stockholm,Sweden,2001264,1000209,2024-05-16T17:32:28.547589Z,Pearly Pies,28,3,84,mastercard,5287105980593305
3000047,Sweet Sinsations,Stockholm,Sweden,2001287,1000120,2024-05-15T08:41:28.406738Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


## C. Full Outer Join Operations

Let's analyze the relationships between dataframes and identify missing data using outer joins

In [0]:
# Let's analyze the relationship between franchises and suppliers using a full outer join
full_join = franchises_df \
    .withColumnRenamed("name", "franchise_name") \
    .join(
        suppliers_df.select("supplierID", col("name").alias("supplier_name")),
        on="supplierID",
        how="full_outer" # Doing outer join
    )

# Find records that would NOT appear in an inner join
# These are records where either franchises or suppliers data is null
non_matching_records = full_join.filter(
        col("franchiseID").isNull() | 
        col("supplier_name").isNull()
    ) \
    .select("franchiseID", "franchise_name", col("supplierID").alias("orphaned_supplier_id"))

display(non_matching_records)

franchiseID,franchise_name,orphaned_supplier_id
3000027,Kanazawa Konfections,4000027
3000028,Batter Up,4000028
3000029,Suita Sweets,4000029
3000030,Caramel Cravings,4000030
3000031,Niigata Nibbles,4000031
3000032,Sweet Temptations,4000032
3000033,Chiba Chews,4000033
3000034,Frosted Fantasies,4000034
3000035,Kumamoto Crumbles,4000035
3000036,Sugar High,4000036


### Using Spark SQL

Let's do this using Spark SQL now....


In [0]:
# Create temporary views
franchises_df.createOrReplaceTempView("franchises")
suppliers_df.createOrReplaceTempView("suppliers")

In [0]:
%sql
-- Let's do our outer join using SQL
SELECT 
    f.franchiseID,
    f.name as franchise_name,
    f.supplierID as orphaned_supplier_id
FROM franchises f
FULL OUTER JOIN suppliers s
ON f.supplierID = s.supplierID
WHERE f.franchiseID IS NULL OR s.name IS NULL

franchiseID,franchise_name,orphaned_supplier_id
3000027,Kanazawa Konfections,4000027
3000028,Batter Up,4000028
3000029,Suita Sweets,4000029
3000030,Caramel Cravings,4000030
3000031,Niigata Nibbles,4000031
3000032,Sweet Temptations,4000032
3000033,Chiba Chews,4000033
3000034,Frosted Fantasies,4000034
3000035,Kumamoto Crumbles,4000035
3000036,Sugar High,4000036


## D. Set Operations

Now let's explore relationships using set operations using the DataFrame API.

In [0]:
# Identify supplier IDs in each DataFrame
franchise_suppliers = franchises_df.select("supplierID").distinct()
all_suppliers = suppliers_df.select("supplierID").distinct()

# Find supplierIDs that are in franchises_df but not in suppliers_df
franchises_without_valid_suppliers = franchise_suppliers.subtract(all_suppliers)
display(franchises_without_valid_suppliers)

supplierID
4000034
4000044
4000037
4000039
4000047
4000045
4000031
4000028
4000032
4000033


In [0]:
# Find the overlap - suppliers that exist in both tables
common_suppliers = franchise_suppliers.intersect(all_suppliers)
display(common_suppliers)

supplierID
4000022
4000021
4000005
4000003
4000004
4000009
4000015
4000019
4000013
4000026


## Key Takeaways

1. **Join Strategy**
   - Use inner joins where keys exist in all dataframes
   - Use outer joins where there is a possibility that keys don't exist in both dataframes
   - Handle column name conflicts

2. **Performance Optimization**
   - Filter before joining
   - Project only needed columns
   - Handle skewed keys appropriately
   - Reference the smaller dataframe first; or
   - Use broadcast joins for small tables


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
